In [122]:
# from pystac_client import Client
# from dask.distributed import Client as DaskClient
# from odc.stac import load, configure_s3_access

import geopandas as gpd
import joblib
import numpy as np
import odc.geo 
import pandas as pd
from shapely import geometry
from sklearn.ensemble import RandomForestClassifier
import rioxarray 
import xarray as xr
from utils import load_data, safe_divide

In [123]:
%reload_ext autoreload
%autoreload 2

## Find and load S2 data

Load data and set up your array to use for prediction

In [124]:
# Create local dask cluster to improve data load time. Only run this once.
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='16GB')

# Configure S3 access. Cloud defaults is an optimisation, while requester pays is required for Landsat
configure_s3_access(cloud_defaults=True, requester_pays=True)

/srv/conda/envs/notebook/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37209 instead
  warnings.warn(


In [156]:
# Configure some things up front
chunks = dict(x=2048, y=2048)
datetime = "2022"

# bbox_fiji = [177.14, -18.41, 179.99, -16.0]
# bbox = bbox_fiji

bbox = [178.14, -17.81, 178.50, -17.4]
bbox_geometry = geometry.box(*bbox)

gdf = gpd.GeoDataFrame({'geometry': [bbox_geometry]}, crs='EPSG:4326')
gdf.explore()

In [157]:
merged = load_data(bbox, chunks=dict(x=2048, y=2048), resolution=10)

In [158]:
merged

<xarray.Dataset> Size: 3GB
Dimensions:        (y: 4760, x: 4008)
Coordinates:
  * y              (y) float64 38kB -1.955e+06 -1.955e+06 ... -2.002e+06
  * x              (x) float64 32kB 3.133e+06 3.133e+06 ... 3.173e+06 3.173e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2022-01-01
Data variables: (12/28)
    blue           (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    green          (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    red            (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    rededge1       (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    rededge2       (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    rededge3       (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    ...             ...
    slope          (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    aspect         (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    elevation_std  (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    mean_vv        (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    mean_vh        (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    mean_vv_vh     (y, x) float32 76MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>

## Train and predict

When you change your training data, you can re-train and predict here.

In [161]:
training_file = "training_data/mred-tdata.geojson"

tdata = gpd.read_file(training_file, bbox=bbox_geometry)
tdata = tdata.set_geometry("geometry")
tdata = tdata.to_crs(merged.rio.crs)

print(tdata.crs)
print(merged.rio.crs)
if tdata.crs != merged.rio.crs:
    tdata = tdata.to_crs(merged.rio.crs)

tdata.explore()

PROJCS["WGS 84 / PDC Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",150],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3832"]]
EPSG:3832


In [162]:
len(tdata)

197

In [163]:
tdata['lulc_code'].value_counts().sort_index()

lulc_code
1    55
2    58
3    11
4    13
5     9
6     9
7    32
8    10
Name: count, dtype: int64

In [165]:
# 1. Ensure both tables are aligned correctly using the shared 'index' column
# This prevents positional or index-label mismatches from creating NaN rows
training_array = pd.merge(tdata, pt_values, on="index", suffixes=('_tdata', '_pt'))

# 2. If tdata is a GeoDataFrame, ensure geometry is preserved from tdata
if not isinstance(training_array, gpd.GeoDataFrame):
    training_array = gpd.GeoDataFrame(training_array, geometry=tdata.geometry, crs=tdata.crs)

# 3. Reproject the spatial GeoDataFrame to WGS84 (EPSG:4326) safely
training_array = training_array.to_crs(4326)

# 4. Drop unwanted metadata, coordinate, and duplicate columns cleanly
cols_to_drop = [
    "y_tdata", "x_tdata", "y_pt", "x_pt", "y", "x",
    "spatial_ref", "time", "fid", "index", "lulc_class",
    "path", "geometry", "layer", "id", "Unnamed: 0"
]

# Only drop columns that actually exist in the dataframe to avoid KeyErrors
existing_cols_to_drop = [col for col in cols_to_drop if col in training_array.columns]
training_array = training_array.drop(columns=existing_cols_to_drop)

# 5. Check final shape and clean NaNs safely
print("Final training array shape:", training_array.shape)
training_array = training_array.dropna(subset=['lulc_code'])

Final training array shape: (197, 26)


In [151]:
len(training_array)

197

In [152]:
training_array

,lulc_code,blue,green,red,rededge1,rededge2,rededge3,nir,swir16,swir22,...,savi,bsi,ndmi,ndbi,ndwi,ndre,elevation,slope,aspect,elevation_std
0,1,811,934,936,1194,1518,1656,1236,1195,873,...,0.207135,0.020105,0.016865,0.119469,30.061751,0.017284,5.000000,11.388381,285.587036,0.000000
1,2,413,643,323,1019,3803,5025,5047,1926,839,...,1.319430,8.084706,0.447584,0.577354,10.743761,0.664029,180.775040,28.120783,146.664017,7.844488
2,8,647,923,664,1286,2815,3302,3492,1822,1058,...,1.020570,9.642717,0.314264,0.372836,14.262061,0.461699,15.229146,5.488382,231.532440,1.335231
3,8,837,874,831,1102,1722,1949,1155,1298,895,...,0.244651,0.033244,26.658377,0.219547,32.161163,0.023482,14.684795,10.203218,261.407593,0.922163
4,2,423,725,358,1134,3868,4925,5089,2155,986,...,1.302708,7.792773,0.405025,0.546581,10.521500,0.635546,215.913071,11.223964,292.107697,0.812147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192,4,632,987,679,1456,3101,3559,3576,2304,1166,...,1.021149,8.943262,0.216327,0.360983,13.795091,0.421304,227.509033,3.215426,33.427856,2.086269
193,4,605,1082,655,1579,3587,4167,4162,2388,1133,...,1.091956,8.170551,0.270840,0.388695,11.909992,0.449922,247.375854,11.339568,308.213562,2.653316
194,4,677,1145,866,1724,3244,3721,3812,2578,1316,...,0.944534,8.129459,0.193114,0.305958,12.682873,0.377168,358.052979,38.245377,242.712311,10.455167
195,4,721,1152,838,1768,3100,3477,3526,2641,1349,...,0.923817,8.383122,0.143506,0.273624,13.501924,0.332074,351.652405,7.140323,81.491882,1.017761


In [153]:
classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42,
)

training_data = np.array(training_array)[:, 1:]
classes = np.array(training_array)[:, 0]

model = classifier.fit(training_data, classes)


In [154]:
joblib.dump(model, "test_model.dump")

['test_model.dump']

In [155]:
# Print feature importances against column headings
fields_importances = sorted(
    zip(training_array.columns[1:], classifier.feature_importances_),
    key=lambda x: x[1],
    reverse=True,
)

for i in fields_importances:
    # Format as a table to 2 decinal places
    print(f"{i[0]:<11}| {i[1]:.2f}")


evi        | 0.10
nir        | 0.09
savi       | 0.09
elevation  | 0.07
ndvi       | 0.06
swir22     | 0.06
ndwi       | 0.06
ndre       | 0.06
rededge2   | 0.05
red        | 0.05
ndbi       | 0.05
swir16     | 0.04
blue       | 0.04
rededge3   | 0.04
smad       | 0.03
rededge1   | 0.03
slope      | 0.02
ndmi       | 0.01
green      | 0.01
mndwi      | 0.01
emad       | 0.01
bcmad      | 0.01
elevation_std| 0.01
bsi        | 0.00
aspect     | 0.00
